<a href="https://colab.research.google.com/github/fabriciosantana/mcdia/blob/main/10-adap/assignments/04/m4_exercicio_agente_compras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Instituto Brasileiro de Ensino, Desenvolvimento e Pesquisa**

Programa de Pós-Graduação em Administração Pública — Mestrado Profissional

**Disciplina:** Auditoria de dados e accountability com python

**Professor:** Aloísio Dourado Neto

**Atividade:** Exercício do Módulo IV — Agente analisador de compras públicas

**Grupo:** Fabricio Santana e Giovanni Brígido

# Exercício do Módulo IV
Este notebook implementa um agente para analisar indícios de sobrepreço em compras públicas a partir de itens semanticamente similares.

A regra adotada é: há **possível sobrepreço** quando o preço estimado do item alvo é estritamente maior que `1,10 × média + 0,50 × desvio padrão` dos preços de referência.

In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Execução local detectada; o Google Drive não será montado.')

Execução local detectada; o Google Drive não será montado.


In [2]:
from pathlib import Path

DATA_DIR = Path('./compras/')
if not DATA_DIR.exists():
    DATA_DIR = Path('compras')
if not DATA_DIR.exists():
    DATA_DIR = Path('/content/drive/MyDrive/Aulas/Aulas IDP/2026/Auditoria de Dados e Accountability/modulo IV - Grafos e IA/compras')

csv_file = DATA_DIR / 'itens_compras.csv'

print('Arquivo de dados (customizar se necessário): ', csv_file)


Arquivo de dados (customizar se necessário):  compras/itens_compras.csv


In [3]:
import pandas as pd

df_compras = pd.read_csv(csv_file, sep=";")
display(df_compras.head())

,ID_COMPRA,NUMERO_UASG,NUMERO_COMPRA,ANO_COMPRA,OBJETO,CHAVE_COMPRA_PNCP,ID_ITEM,numero_item,descricao,descricao_detalhada,unidade_fornecimento,valor_estimado,quantidade_solicitada,orcamento_sigiloso,codigo_item_catalogo,tipo_item_catalogo
0,696890,170209,68,2026,Fornecimento de materiais de cuidado dos cães ...,39446000014110005372026,7773562,1,Vacina,"aplicação*: uso veterinário, forma farmacêutic...",Doses,132.50,3,N,439562,M
1,693455,927996,53,2026,Aquisição de MEDICAMENTOS/SUPLEMENTOS CONTROLA...,1695842500014810003442026,7717753,40,Fenobarbital Sódico,dosagem: 100 CONFORME DESCRITO DO TERMO DE REF...,Comprimido,0.07,509600,S,267660,M
2,693280,155124,34,2026,"Aquisição de Medicamentos Anti-inflamatórios, ...",1512643700030510029692026,7713479,11,Tramadol Cloridrato,dosagem: 50,Cápsula,0.15,1750,S,268534,M
3,693657,927502,356,2026,Aquisição dos medicamentos VACINA PARA IMUNOTE...,73306200010210003352026,7719485,1,Vacina,"composição 1: extrato alérgeno de ácaros, form...",Frasco,281.22,10,N,483360,M
4,693455,927996,53,2026,Aquisição de MEDICAMENTOS/SUPLEMENTOS CONTROLA...,1695842500014810003442026,7717776,63,Levomepromazina,dosagem: 25 CONFORME DESCRITO DO TERMO DE REFE...,Comprimido,0.59,609700,S,268128,M


In [4]:
compras_metadata = df_compras.to_json(orient='records', lines=True, force_ascii=False).splitlines()

print(f"Total de {len(compras_metadata)} linhas JSON geradas.")
print("Primeiras 5 linhas JSON:")
for i, line in enumerate(compras_metadata[:5]):
    print(f"Linha {i+1}: {line}")

Total de 333 linhas JSON geradas.
Primeiras 5 linhas JSON:
Linha 1: {"ID_COMPRA":696890,"NUMERO_UASG":170209,"NUMERO_COMPRA":68,"ANO_COMPRA":2026,"OBJETO":"Fornecimento de materiais de cuidado dos cães de faro da RFB, como ração, vacinas, vermífugos, carrapaticidas, suplementos, medicamentos, vitaminas e materiais de higiene, conforme necessidade ALF\/AEG","CHAVE_COMPRA_PNCP":39446000014110005372026,"ID_ITEM":7773562,"numero_item":1,"descricao":"Vacina","descricao_detalhada":"aplicação*: uso veterinário, forma farmacêutica: suspensão injetável, outros componentes: b. bronchiseptica, tipo: inativada ","unidade_fornecimento":"Doses","valor_estimado":132.5,"quantidade_solicitada":3,"orcamento_sigiloso":"N","codigo_item_catalogo":439562,"tipo_item_catalogo":"M"}
Linha 2: {"ID_COMPRA":693455,"NUMERO_UASG":927996,"NUMERO_COMPRA":53,"ANO_COMPRA":2026,"OBJETO":"Aquisição de MEDICAMENTOS\/SUPLEMENTOS CONTROLADOS PELA PORTARIA 344\/98 destinados as unidades de saúde pertencentes a Rede Hospitala

In [5]:
compras_indice = df_compras[["descricao",
                             "descricao_detalhada",
                             "unidade_fornecimento",
                             "codigo_item_catalogo"]].to_json(orient='records', lines=True, force_ascii=False).splitlines()

print(f"Total de {len(compras_indice)} linhas JSON geradas.")
print("Primeiras 5 linhas JSON:")
for i, line in enumerate(compras_indice[:5]):
    print(f"Linha {i+1}: {line}")

Total de 333 linhas JSON geradas.
Primeiras 5 linhas JSON:
Linha 1: {"descricao":"Vacina","descricao_detalhada":"aplicação*: uso veterinário, forma farmacêutica: suspensão injetável, outros componentes: b. bronchiseptica, tipo: inativada ","unidade_fornecimento":"Doses","codigo_item_catalogo":439562}
Linha 2: {"descricao":"Fenobarbital Sódico","descricao_detalhada":"dosagem: 100 CONFORME DESCRITO DO TERMO DE REFERÊNCIA: ","unidade_fornecimento":"Comprimido","codigo_item_catalogo":267660}
Linha 3: {"descricao":"Tramadol Cloridrato","descricao_detalhada":"dosagem: 50 ","unidade_fornecimento":"Cápsula","codigo_item_catalogo":268534}
Linha 4: {"descricao":"Vacina","descricao_detalhada":"composição 1: extrato alérgeno de ácaros, forma farmacêutica 1: injetável, tipo 1: padronizado ","unidade_fornecimento":"Frasco","codigo_item_catalogo":483360}
Linha 5: {"descricao":"Levomepromazina","descricao_detalhada":"dosagem: 25 CONFORME DESCRITO DO TERMO DE REFERÊNCIA: ","unidade_fornecimento":"Compr

## Criando uma Base de Conhecimento Vetorial

Para criar uma base de conhecimento vetorial, seguiremos estes passos:
1.  **Instalar bibliotecas necessárias**: `chromadb` e `openai`.
2.  **Gerar embeddings**: Converter esses pedaços em representações vetoriais numéricas usando o modelo de embedding da OpenAI.
3.  **Criar o armazenamento vetorial**: Armazenar esses embeddings em uma instância ChromaDB, que permite buscas eficientes por similaridade.

In [19]:
%pip install -q chromadb openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [20]:
%pip install -q langchain langchain-openai langchain-community langchain-chroma langgraph

Note: you may need to restart the kernel to use updated packages.


In [6]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [7]:
env_path = Path('.env')

import json
from dotenv import load_dotenv
load_dotenv(env_path)

from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

documents_from_json = [
    Document(page_content=line, metadata=json.loads(meta))
    for line, meta in zip(compras_indice, compras_metadata)
]

db = Chroma.from_documents(documents_from_json, embeddings, persist_directory='vectordb')

print("Base vetorial criada com sucesso!")

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

A base de dados vetorial `db` está agora pronta. Você pode usá-la para buscas por similaridade ou para recuperar informações relevantes.

## Compras Novas
As compras abaixo terão seus preços avaliados

In [ ]:
csv_file_path = DATA_DIR / 'compras_novas.csv'
df_novas_compras = pd.read_csv(csv_file_path, sep=";")
df_novas_compras.head()

## Agente verificador de compras

### Estado do agente

In [ ]:
from typing import TypedDict, List, Optional, Annotated
from langchain_core.messages import BaseMessage, FunctionMessage
from langgraph.graph import add_messages
from pydantic import BaseModel

# Definir o modelo Pydantic para os detalhes do item
class ItemDetalhes(BaseModel):
    CHAVE_COMPRA_PNCP: str
    OBJETO: str
    numero_item: int
    descricao: str
    descricao_detalhada: str
    unidade_fornecimento: str
    valor_estimado: float
    quantidade_solicitada: int
    # Adicionando campos de metadados para melhor rastreamento
    NUMERO_UASG: Optional[str] = None
    NUMERO_COMPRA: Optional[str] = None
    ANO_COMPRA: Optional[int] = None

class ItemNaoSimilar(BaseModel):
    CHAVE_COMPRA_PNCP: str
    OBJETO: str
    numero_item: int
    descricao: str
    descricao_detalhada: str
    unidade_fornecimento: str
    valor_estimado: float
    quantidade_solicitada: int
    # Adicionando campos de metadados para melhor rastreamento
    NUMERO_UASG: Optional[str] = None
    NUMERO_COMPRA: Optional[str] = None
    ANO_COMPRA: Optional[int] = None
    JUSTIFICATIVA_DE_NAO_SIMILARIDADE: Optional[str]

# Definir o estado do agente
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


### Definição do agente

In [ ]:
from langchain.tools import tool
import pandas as pd
from typing import Optional

@tool
def get_pncp_key_by_index(index: int) -> Optional[ItemDetalhes]:
    """Retorna um objeto `ItemDetalhes` de um registro no arquivo 'compras_novas.csv' pelo índice.

    Args:
        index (int): O índice baseado em zero do registro no DataFrame para o qual o objeto `ItemDetalhes` deve ser retornado.

    Returns:
        Optional[ItemDetalhes]: O objeto `ItemDetalhes` correspondente ao índice fornecido, ou `None` se o índice for inválido,
             o arquivo não for encontrado, uma coluna necessária não existir ou ocorrer outro erro.
    """
    csv_file_path = DATA_DIR / 'compras_novas.csv'
    try:
        df_novas_compras = pd.read_csv(csv_file_path, sep=";")
        if 0 <= index < len(df_novas_compras):
            row = df_novas_compras.loc[index]
            item_detalhes = ItemDetalhes(
                CHAVE_COMPRA_PNCP=row['CHAVE_COMPRA_PNCP'],
                OBJETO=row['OBJETO'],
                numero_item=int(row['numero_item']),
                descricao=row['descricao'],
                descricao_detalhada=row['descricao_detalhada'],
                unidade_fornecimento=row['unidade_fornecimento'],
                valor_estimado=float(row['valor_estimado']),
                quantidade_solicitada=int(row['quantidade_solicitada']),
                NUMERO_UASG= str(row['NUMERO_UASG']),
                NUMERO_COMPRA=str(row['NUMERO_COMPRA']),
                ANO_COMPRA=int(row['ANO_COMPRA'])
            )
            return item_detalhes
        else:
            print(f"Índice {index} fora dos limites do DataFrame.")
            return None
    except FileNotFoundError:
        print(f"Arquivo não encontrado: {csv_file_path}")
        return None
    except KeyError as e:
        print(f"Coluna não encontrada no arquivo CSV ou nome incorreto: {e}")
        return None
    except Exception as e:
        print(f"Ocorreu um erro ao carregar ou processar o arquivo: {e}")
        return None


## **Ponto de Mudança**
Substitua a Tool abaixo por uma tool que retorne a media e o desvio padrão da lista de preços.

Não se esqueça de ajustar a Docstring de acordo

In [ ]:
@tool
def calculate_price_statistics(prices: list[float]) -> dict[str, float]:
    """Calcula a média e o desvio padrão populacional dos preços de referência.

    Args:
        prices: Lista não vazia de preços numéricos e finitos.

    Returns:
        Dicionário com as chaves ``media`` e ``desvio_padrao``.

    Raises:
        ValueError: Se a lista estiver vazia ou contiver valor não numérico ou não finito.
    """
    import math
    import statistics

    if not prices:
        raise ValueError("A lista de preços de referência não pode ser vazia.")
    if any(isinstance(price, bool) or not isinstance(price, (int, float)) for price in prices):
        raise ValueError("Todos os preços devem ser numéricos.")

    normalized_prices = [float(price) for price in prices]
    if not all(math.isfinite(price) for price in normalized_prices):
        raise ValueError("Todos os preços devem ser finitos.")

    return {
        "media": statistics.mean(normalized_prices),
        "desvio_padrao": statistics.pstdev(normalized_prices),
    }


## **Nova Tool**
Adicione na célula abaixo, uma nova tool que verifica sobrepreço conforme as regras estabelecidas. Deve receber o preço do alvo, a média e o desvio padrão dos preços de referência e retornar "Possível sobrepreço" ou "Sobrepreço não identificado"

In [ ]:
@tool
def check_overpricing(
    target_price: float,
    mean_price: float,
    standard_deviation: float,
) -> str:
    """Verifica indício de sobrepreço pela regra definida na atividade.

    O limite é ``1.10 * mean_price + 0.50 * standard_deviation``. Como a regra
    exige que o preço seja maior que o limite, a igualdade não indica sobrepreço.

    Args:
        target_price: Preço estimado do item alvo.
        mean_price: Média dos preços dos itens de referência.
        standard_deviation: Desvio padrão dos preços de referência.

    Returns:
        ``"Possível sobrepreço"`` se o preço alvo superar o limite; caso
        contrário, ``"Sobrepreço não identificado"``.
    """
    import math

    values = (target_price, mean_price, standard_deviation)
    if any(isinstance(value, bool) or not isinstance(value, (int, float)) for value in values):
        raise ValueError("Preço alvo, média e desvio padrão devem ser numéricos.")
    if not all(math.isfinite(float(value)) for value in values):
        raise ValueError("Preço alvo, média e desvio padrão devem ser finitos.")
    if mean_price < 0 or standard_deviation < 0:
        raise ValueError("Média e desvio padrão não podem ser negativos.")

    threshold = 1.10 * float(mean_price) + 0.50 * float(standard_deviation)
    return (
        "Possível sobrepreço"
        if float(target_price) > threshold
        else "Sobrepreço não identificado"
    )


### Realizando Geração Aumentada por Recuperação (RAG)

Agora vamos integrar a base de conhecimento vetorial com um modelo de linguagem para realizar o RAG. Isso permitirá que o modelo de linguagem use o contexto recuperado dos seus documentos para gerar respostas mais precisas e informadas à sua pergunta.

In [ ]:
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from typing import List # Ensure List is imported
from operator import itemgetter

# Initialize the LLM (Large Language Model)
llm = ChatOpenAI(model="gpt-5.4", temperature=0.1)

# Create a retriever from your ChromaDB instance, returning 15 items
retriever = db.as_retriever(search_kwargs={'k': 15})

# 1. Definir o modelo Pydantic para a lista de saída estruturada
class ListaDeItens(BaseModel):
    """Representa uma lista de itens de compra analisados encontrados na base de conhecimento."""
    itens_similares: List[ItemDetalhes]
    itens_nao_similares: List[ItemNaoSimilar]

# 2. Configurar o LLM para retornar saída estruturada
structured_llm = llm.with_structured_output(ListaDeItens)

# Helper function to format retrieved documents into a single string, including metadata
def format_docs_with_metadata(docs):
    formatted_docs = []
    for doc in docs:
        content = doc.page_content
        metadata = doc.metadata
        formatted_docs.append(f"--- Documento de Contexto ---\nConteúdo: {content}\nMetadados: {metadata}\n--------------------------")
    return "\n\n".join(formatted_docs)

# 3. Criar um prompt específico para RAG com saída estruturada
rag_prompt_structured = ChatPromptTemplate.from_messages([
    ("system", """Você é um assistente especialista em compras públicas. Sua tarefa é encontrar itens de compra semanticamente similares ao 'item_alvo' fornecido,
                  EXCLUSIVAMENTE dentro do contexto de compras públicas disponibilizado. Para cada item similar encontrado no contexto, extraia todas as informações necessárias para preencher o modelo ItemDetalhes.
                  Os metadados dos documentos de contexto estão disponíveis na seção 'Metadados:' de cada 'Documento de Contexto'.
                  Certifique-se de extrair `CHAVE_COMPRA_PNCP`, `NUMERO_UASG`, `NUMERO_COMPRA` e `ANO_COMPRA` dos metadados dos documentos de contexto, se disponíveis.
                  Voce deve retornar 2 listas ESTRITAMENTE seguindo o esquema `ListaDeItens`:
                    A: lista de itens similares - dos itens presentes no contexto, relacione aqui aqueles itens que são
                       similares ao item alvo.
                    B: Lista de itens não similares - dos itens presentes no contexto, relacione aqui aqueles que não sao similares para fins de cmparação de preço.    DOS ESTRITAMENTE seguindo o esquema `ListaDeItens`.
                  Não inclua nenhum item que não esteja explicitamente no contexto.
                  Se nenhum item similar for encontrado, retorne uma lista vazia.
                  Contexto de compras (com conteúdo e metadados):\n{context}
               """),
    ("user", "Com base no contexto, encontre todos os itens de compra similares ao seguinte item alvo: {input}"),
])

# 4. Criar uma cadeia de documentos estruturada usando LCEL
chain_structured = (
     rag_prompt_structured # Pass formatted context and input to the prompt
    | structured_llm # Pass the prompted input to the structured LLM
)

# A `retrieval_chain_structured` será removida e a lógica de recuperação será feita explicitamente na ferramenta.

@tool
def find_similar_items(item_alvo: dict) -> ListaDeItens:
    """Encontra itens de compra similares a um item alvo especificado, utilizando uma base de conhecimento vetorial.

    Args:
        item_alvo (dict): O item alvo para o qual buscar itens similares, contendo as informações necessárias:
                                  CHAVE_COMPRA_PNCP, OBJETO, descricao, descricao_detalhada, unidade_fornecimento, valor_estimado e quantidade_solicitada.

    Returns:
        ListaDeItens: Um objeto contendo uma lista de objetos ItemDetalhes que são considerados similares ao item alvo,
                               extraídos da base de conhecimento. Retorna uma lista vazia se nenhum item similar for encontrado
                               ou em caso de erro.
    """
    # Construir a string de consulta para a cadeia RAG com base nos atributos do item_alvo
    query_string = \
        f"Item alvo para busca de similares: Objeto: {item_alvo['OBJETO']}, " +\
        f"Descrição: {item_alvo['descricao']}, "+\
        f"Descrição Detalhada: {item_alvo['descricao_detalhada']}, "+\
        f"Unidade de Fornecimento: {item_alvo['unidade_fornecimento']}. " +\
        f"Valor Estimado: {item_alvo['valor_estimado']}, Quantidade Solicitada: {item_alvo['quantidade_solicitada']}."


    try:
        #print("Query para o retriever: <"+query_string+">")
        # 1. Recuperar documentos explicitamente
        retrieved_docs = retriever.invoke(query_string)
        #print(f"Documentos recuperados: {len(retrieved_docs)}")

        # 2. Formatar os documentos com metadados
        formatted_docs = format_docs_with_metadata(retrieved_docs)
        #print("Documentos formatados:")
        #print(formatted_docs)

        # 3. Invocar a cadeia de documentos estruturada com os documentos recuperados e a query
        response = chain_structured.invoke({"context": formatted_docs, "input": query_string})
        #print("Response do structured_llm: <"+str(response)+">")

        # A 'response' já é o objeto ListaDeItensSimilares diretamente do structured_llm
        return response
    except Exception as e:
        print(f"Erro ao buscar itens similares: {e}")
        return ListaDeItens(itens_similares=[], itens_nao_similares=[])


## Construindo o Agente de Análise de Preços

### **Mudança**: Ajuste a lista de tools e o prompt para a nova especificação

In [ ]:
from typing import TypedDict, Annotated, List, Optional
from langchain_core.messages import BaseMessage, FunctionMessage, HumanMessage, AIMessage
from langchain_core.runnables import RunnablePassthrough
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode # Remover ToolExecutor da importação

# As classes ItemDetalhes, AgentState e as ferramentas são definidas em células anteriores
# são definidas em células anteriores e devem estar no escopo.
# O 'llm' e o 'retriever' também devem estar definidos em células anteriores.

# 1. Agrupar as ferramentas
tools = [
    get_pncp_key_by_index,
    find_similar_items,
    calculate_price_statistics,
    check_overpricing,
]

# 2. Criar um LLM com capacidade de chamada de ferramentas
agent_llm = llm.bind_tools(tools)

# 3. Definir o nó do agente que invoca o LLM
def agent_runner(state: AgentState):
    system_prompt = """Você é um agente especializado em auditoria de compras públicas. Analise o preço do item alvo executando obrigatoriamente estas etapas, nesta ordem:

1. Use `get_pncp_key_by_index` para obter o item alvo a partir do índice informado.
2. Use `find_similar_items` com os detalhes completos do alvo.
3. Extraia `valor_estimado` somente de `itens_similares`. Não use o preço do próprio item alvo nem os itens de `itens_nao_similares` como referências. Se não houver item similar, informe que não há base suficiente para a análise e encerre sem classificar sobrepreço.
4. Use `calculate_price_statistics` uma única vez com a lista dos preços de referência. Essa ferramenta retorna a média e o desvio padrão populacional.
5. Use `check_overpricing` com o preço estimado do alvo, a média e o desvio padrão retornados. Não faça a classificação por cálculo próprio.
6. Apresente uma conclusão concisa contendo: quantidade de referências, média, desvio padrão, limite calculado (`1,10 × média + 0,50 × desvio padrão`), preço alvo e exatamente uma das classificações retornadas pela ferramenta.

Valores monetários devem ser mostrados com duas casas decimais. Nunca invente itens ou preços e não classifique pela mediana."""
    prompt_template = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("placeholder", "{messages}"),
    ])

    chain = prompt_template | agent_llm

    messages = state['messages']
    # O agente invoca o LLM com as mensagens atuais. ToolNode cuidará dos resultados das ferramentas.
    response = chain.invoke({"messages": messages})
    #print(response)
    return {"messages": [response]}

# 5. Definir a lógica condicional para decidir o próximo passo
def should_continue(state: AgentState) -> str:
    messages = state['messages']
    last_message = messages[-1]
    # Se a última mensagem do agente inclui chamadas de ferramentas, vá para 'call_tool'.
    # Caso contrário, o agente chegou a uma conclusão e deve terminar.
    if last_message.tool_calls:
        return "call_tool"
    else:
        return "end_agent"

# 6. Construir o StateGraph
workflow = StateGraph(AgentState)

workflow.add_node("agent_node", agent_runner)
# Usar ToolNode padrão para executar as ferramentas
workflow.add_node("call_tool", ToolNode(tools=tools))

workflow.set_entry_point("agent_node")

workflow.add_conditional_edges(
    "agent_node",
    should_continue,
    {
        "call_tool": "call_tool",
        "end_agent": END
    },
)
# Após a execução da ferramenta, o fluxo retorna ao agent_node para o LLM processar o resultado da ferramenta
workflow.add_edge('call_tool', 'agent_node')

# 7. Compilar o gráfico em um agente executável
agent_executor = workflow.compile()

### Visualizando o Grafo

In [ ]:
from IPython.display import Image
display(Image(agent_executor.get_graph().draw_mermaid_png()))

### Executando o Agente

Agora você pode interagir com o agente fornecendo um índice para o item alvo. O agente usará suas ferramentas para realizar a análise e retornar o resultado.

In [ ]:
def run_agent_analysis(item_index: int):
    # Initialize the state for the agent with the user's request
    # The system message is already part of the agent_prompt defined above.
    # The agent_runner expects a list of messages for its input.
    # So, we start with a HumanMessage requesting the item at the given index.

    initial_input = {
        "messages": [HumanMessage(content=f"Analise o item com índice {item_index}.")]
    }

    # The AgentState needs to be correctly initialized for the graph.
    # The initial_input will be merged into the AgentState at the start.

    print(f"Iniciando análise para o item com índice: {item_index}")

    # Stream events from the agent to see its steps
    for s in agent_executor.stream(initial_input):
        if "agent_node" in s:
            for message in s["agent_node"]["messages"]:
                print(f"Agent: {message.content}")
                if message.tool_calls:
                    for tc in message.tool_calls:
                        # Acessar 'name' e 'args' como chaves de dicionário, pois 'tc' parece ser um dicionário
                        print(f"Agent calls tool: {tc['name']} with args {tc['args']}")
        elif "call_tool" in s:
            for message in s["call_tool"]["messages"]:
                print(f"Tool Output: {message.content}")
        elif END in s:
            final_state = s[END]
            final_message = final_state['messages'][-1]
            print("\n--- Análise Finalizada ---")
            print(final_message.content)


# Exemplo de uso do agente:
# run_agent_analysis(item_index=0) # Altere o índice conforme necessário para testar

In [ ]:
run_agent_analysis(item_index=57)